# 카테고리별 TF-IDF Vectorizer 학습 (핵심어 추출용)
카테고리마다 별도 vectorizer를 학습해서 카테고리 내 IDF 기준으로 핵심어를 뽑습니다.

In [ ]:
!apt-get install -y openjdk-11-jdk -qq
!pip install konlpy scikit-learn joblib -q

In [ ]:
from google.colab import files
uploaded = files.upload()  # train_data.json 업로드
import os
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.rename('train_data.json', 'data/train_data.json')

In [ ]:
import json
import joblib
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from konlpy.tag import Okt

okt = Okt()

def extract_nouns(text):
    return " ".join(okt.nouns(text))


print("데이터 로딩...")
with open('data/train_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 카테고리별로 분리
category_texts = defaultdict(list)
for d in data:
    category_texts[d['topic']].append(d['text'])

print(f"카테고리: {list(category_texts.keys())}")
for cat, texts in category_texts.items():
    print(f"  {cat}: {len(texts)}개")


print("\n명사 추출 및 vectorizer 학습 중...")
for category, texts in category_texts.items():
    print(f"\n[{category}] 명사 추출 중... ({len(texts)}개)")
    corpus = [extract_nouns(t) for t in texts]

    print(f"[{category}] TF-IDF 학습...")
    vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
    vectorizer.fit(corpus)

    save_path = f"models/tfidf_{category}.pkl"
    joblib.dump(vectorizer, save_path)
    print(f"[{category}] 저장 완료 → {save_path}")

print("\n전체 완료!")

In [ ]:
# 카테고리별 pkl 압축해서 한번에 다운로드
import shutil
from google.colab import files

shutil.make_archive('tfidf_vectorizers', 'zip', 'models')
files.download('tfidf_vectorizers.zip')